In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, ConfusionMatrixDisplay,
                              classification_report)
import time, os, urllib.request, zipfile

tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

In [ ]:
DATA_URL = "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"
ZIP_PATH = "har.zip"
EXTRACT_DIR = "har_data"

if not os.path.exists(EXTRACT_DIR):
    print("Downloading UCI HAR dataset...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)
    # The zip contains a nested "UCI HAR Dataset.zip" — extract that too
    inner_zip = os.path.join(EXTRACT_DIR, "UCI HAR Dataset.zip")
    if os.path.exists(inner_zip):
        with zipfile.ZipFile(inner_zip, "r") as z:
            z.extractall(EXTRACT_DIR)

BASE = os.path.join(EXTRACT_DIR, "UCI HAR Dataset")
print(os.listdir(BASE))

In [ ]:
SIGNALS = [
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z",
]

ACTIVITY_NAMES = {
    1: "WALKING", 2: "WALKING_UPSTAIRS", 3: "WALKING_DOWNSTAIRS",
    4: "SITTING", 5: "STANDING", 6: "LAYING",
}

def load_signals(split):
    """Returns X of shape (N, 128, 9) for the given split ('train' or 'test')."""
    signal_dir = os.path.join(BASE, split, "Inertial Signals")
    channels = []
    for sig in SIGNALS:
        fname = os.path.join(signal_dir, f"{sig}_{split}.txt")
        channels.append(np.loadtxt(fname))          # (N, 128)
    X = np.stack(channels, axis=-1)                  # (N, 128, 9)
    return X

def load_labels(split):
    fname = os.path.join(BASE, split, f"y_{split}.txt")
    return np.loadtxt(fname).astype(int)              # 1..6

X_train_full = load_signals("train")
y_train_full = load_labels("train")
X_test_full  = load_signals("test")
y_test_full  = load_labels("test")

print("Raw train:", X_train_full.shape, y_train_full.shape)
print("Raw test :", X_test_full.shape, y_test_full.shape)

In [ ]:
N_SUBSET = 2400   # within the recommended 1500-3000 range, divisible by 6 classes

# Stratified subset preserving all 6 classes with balanced representation
idx_subset, _ = train_test_split(
    np.arange(len(X_train_full)),
    train_size=N_SUBSET / len(X_train_full),
    stratify=y_train_full,
    random_state=42,
)
X_sub = X_train_full[idx_subset]
y_sub = y_train_full[idx_subset]

# Label encoding (0..5)
le = LabelEncoder()
y_sub_enc = le.fit_transform(y_sub)
y_test_enc = le.transform(y_test_full)

# 70/15/15 split — first carve out test(15%) & val(15%) from the labelled subset+held-out test
X_temp, X_val, y_temp, y_val = train_test_split(
    X_sub, y_sub_enc, test_size=0.15, stratify=y_sub_enc, random_state=42
)
X_train, X_test_int, y_train, y_test_int = train_test_split(
    X_temp, y_temp, test_size=0.15/0.85, stratify=y_temp, random_state=42
)

# Final independent test set: combine the held-out internal test split with the official UCI test partition
X_test = np.concatenate([X_test_int, X_test_full], axis=0)
y_test = np.concatenate([y_test_int, y_test_enc], axis=0)

print("Training  :", X_train.shape)
print("Validation:", X_val.shape)
print("Testing   :", X_test.shape)

In [ ]:
# Normalize per-channel using TRAINING statistics only
n_train, T, F = X_train.shape
scaler = StandardScaler()
scaler.fit(X_train.reshape(-1, F))

def apply_scaler(X):
    shp = X.shape
    return scaler.transform(X.reshape(-1, F)).reshape(shp)

X_train_n = apply_scaler(X_train)
X_val_n   = apply_scaler(X_val)
X_test_n  = apply_scaler(X_test)

NUM_CLASSES = len(le.classes_)

print("Input tensor shape:")
print("  Training  :", X_train_n.shape)
print("  Validation:", X_val_n.shape)
print("  Testing   :", X_test_n.shape)
print("Number of classes:", NUM_CLASSES)
print("Number of features per time step:", F)
print("Sequence length:", T)

# Class distribution check
print("\nClass distribution (train):", np.bincount(y_train))
print("Class distribution (val)  :", np.bincount(y_val))
print("Class distribution (test) :", np.bincount(y_test))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
example_classes = [0, 3, 5]   # e.g. WALKING-like, SITTING-like, LAYING-like after encoding
channel_idx = [0, 3, 6]        # one channel from each sensor group
channel_labels = [SIGNALS[i] for i in channel_idx]

for ax, cls in zip(axes, example_classes):
    sample_idx = np.where(y_train == cls)[0][0]
    for ch, lbl in zip(channel_idx, channel_labels):
        ax.plot(X_train_n[sample_idx, :, ch], label=lbl)
    activity_name = le.inverse_transform([cls])[0]
    ax.set_title(f"Activity: {ACTIVITY_NAMES[activity_name]}")
    ax.set_ylabel("Sensor value")
    ax.legend(loc="upper right", fontsize=8)
axes[-1].set_xlabel("Time step (1-128)")
plt.tight_layout()
plt.show()

In [ ]:
x = [0.5, 0.7, 0.2]
h0, Wx, Wh, b = 0.0, 0.5, 0.8, 0.1

h = h0
hs = []
for xt in x:
    h = np.tanh(Wx * xt + Wh * h + b)
    hs.append(h)

for t, h_t in enumerate(hs, start=1):
    print(f"h{t} = {h_t:.6f}")

In [ ]:
def build_model(cell_type, seq_len=128, num_features=9, units=32, num_classes=NUM_CLASSES):
    layer_map = {
        "RNN": layers.SimpleRNN,
        "LSTM": layers.LSTM,
        "GRU": layers.GRU,
    }
    RecurrentLayer = layer_map[cell_type]

    model = models.Sequential([
        layers.Input(shape=(seq_len, num_features)),
        RecurrentLayer(units),
        layers.Dropout(0.2),
        layers.Dense(16, activation="relu"),
        layers.Dense(num_classes, activation="softmax"),
    ], name=f"{cell_type}_classifier")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

build_model("LSTM").summary()

In [ ]:
BATCH_SIZE = 32
EPOCHS = 30

histories = {}
trained_models = {}
train_times = {}

for cell_type in ["RNN", "LSTM", "GRU"]:
    print(f"\n=== Training {cell_type} ===")
    model = build_model(cell_type)
    es = callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)

    start = time.time()
    history = model.fit(
        X_train_n, y_train,
        validation_data=(X_val_n, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[es],
        verbose=1,
    )
    elapsed = time.time() - start

    histories[cell_type] = history
    trained_models[cell_type] = model
    train_times[cell_type] = elapsed
    print(f"{cell_type} training time: {elapsed:.2f}s")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for i, cell_type in enumerate(["RNN", "LSTM", "GRU"]):
    h = histories[cell_type].history
    axes[0, i].plot(h["loss"], label="Train Loss")
    axes[0, i].plot(h["val_loss"], label="Val Loss")
    axes[0, i].set_title(f"{cell_type} — Loss")
    axes[0, i].set_xlabel("Epoch"); axes[0, i].set_ylabel("Loss"); axes[0, i].legend()

    axes[1, i].plot(np.array(h["accuracy"]) * 100, label="Train Acc")
    axes[1, i].plot(np.array(h["val_accuracy"]) * 100, label="Val Acc")
    axes[1, i].set_title(f"{cell_type} — Accuracy")
    axes[1, i].set_xlabel("Epoch"); axes[1, i].set_ylabel("Accuracy (%)"); axes[1, i].legend()

plt.tight_layout()
plt.show()

In [ ]:
def count_params(model):
    return model.count_params()

results = {}
predictions = {}

for cell_type, model in trained_models.items():
    y_pred_prob = model.predict(X_test_n, verbose=0)
    y_pred = np.argmax(y_pred_prob, axis=1)
    predictions[cell_type] = y_pred

    results[cell_type] = {
        "Accuracy (%)": accuracy_score(y_test, y_pred) * 100,
        "Macro Precision (%)": precision_score(y_test, y_pred, average="macro") * 100,
        "Macro Recall (%)": recall_score(y_test, y_pred, average="macro") * 100,
        "Macro F1 (%)": f1_score(y_test, y_pred, average="macro") * 100,
        "Parameters": count_params(model),
        "Training Time (s)": train_times[cell_type],
    }

results_df = pd.DataFrame(results).T
results_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
class_names = [ACTIVITY_NAMES[c] for c in le.inverse_transform(range(NUM_CLASSES))]

for ax, cell_type in zip(axes, ["RNN", "LSTM", "GRU"]):
    cm = confusion_matrix(y_test, predictions[cell_type])
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
    ax.set_title(cell_type)

plt.tight_layout()
plt.show()

for cell_type in ["RNN", "LSTM", "GRU"]:
    print(f"\n--- {cell_type} classification report ---")
    print(classification_report(y_test, predictions[cell_type], target_names=class_names))

In [ ]:
comparison_df = pd.DataFrame({
    "Hidden state": ["Yes", "Yes", "Yes"],
    "Cell state":   ["No", "Yes", "No"],
    "Forget gate":  ["No", "Yes", "No"],
    "Input gate":   ["No", "Yes", "No"],
    "Output gate":  ["No", "Yes", "No"],
    "Update gate":  ["No", "No", "Yes"],
    "Reset gate":   ["No", "No", "Yes"],
    "Parameters":   [results["RNN"]["Parameters"], results["LSTM"]["Parameters"], results["GRU"]["Parameters"]],
    "Training time (s)": [round(results[c]["Training Time (s)"], 2) for c in ["RNN", "LSTM", "GRU"]],
    "Test F1-score (%)": [round(results[c]["Macro F1 (%)"], 2) for c in ["RNN", "LSTM", "GRU"]],
}, index=["RNN", "LSTM", "GRU"]).T
comparison_df

In [ ]:
labels = ["RNN", "LSTM", "GRU"]
acc = [results[c]["Accuracy (%)"] for c in labels]
f1 = [results[c]["Macro F1 (%)"] for c in labels]
params_norm = np.array([results[c]["Parameters"] for c in labels])
params_norm = params_norm / params_norm.max() * 100   # normalized for comparability

x = np.arange(len(labels))
width = 0.25

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width, acc, width, label="Accuracy (%)")
ax.bar(x, f1, width, label="Macro F1 (%)")
ax.bar(x + width, params_norm, width, label="Normalized Params (%)")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("Value")
ax.set_title("Model Performance Comparison")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
def truncate_sequence(X, seq_len):
    """Keep the first `seq_len` time steps (consistent truncation)."""
    return X[:, :seq_len, :]

seq_lengths = [32, 64, 128]
seqlen_results = {cell_type: {} for cell_type in ["RNN", "LSTM", "GRU"]}

for T_len in seq_lengths:
    Xtr = truncate_sequence(X_train_n, T_len)
    Xva = truncate_sequence(X_val_n, T_len)
    Xte = truncate_sequence(X_test_n, T_len)

    for cell_type in ["RNN", "LSTM", "GRU"]:
        model = build_model(cell_type, seq_len=T_len)
        es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
        model.fit(Xtr, y_train, validation_data=(Xva, y_val),
                  epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[es], verbose=0)
        y_pred = np.argmax(model.predict(Xte, verbose=0), axis=1)
        f1 = f1_score(y_test, y_pred, average="macro") * 100
        seqlen_results[cell_type][T_len] = f1
        print(f"T={T_len:3d} | {cell_type:4s} | Macro F1 = {f1:.2f}%")

seqlen_df = pd.DataFrame(seqlen_results)
seqlen_df

In [ ]:
plt.figure(figsize=(7, 5))
for cell_type in ["RNN", "LSTM", "GRU"]:
    plt.plot(seq_lengths, [seqlen_results[cell_type][t] for t in seq_lengths],
             marker="o", label=cell_type)
plt.xlabel("Sequence length (T)")
plt.ylabel("Test Macro F1-score (%)")
plt.title("Sequence Length vs. Test F1-score")
plt.xticks(seq_lengths)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Mount Drive if your videos live there (optional)
# from google.colab import drive
# drive.mount('/content/drive')

VIDEO_ROOT = "ucf_subset"          # <-- point this at your downloaded subset
SELECTED_CLASSES = ["Basketball", "Biking", "Walking", "Running", "TennisSwing"]
FRAMES_PER_VIDEO = 10
FRAME_SIZE = (224, 224)

In [ ]:
import cv2

def sample_frames(video_path, n_frames=FRAMES_PER_VIDEO, size=FRAME_SIZE):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return None
    idxs = np.linspace(0, max(total - 1, 0), n_frames).astype(int)
    frames = []
    for i in range(total):
        ret, frame = cap.read()
        if not ret:
            break
        if i in idxs:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, size)
            frames.append(frame)
    cap.release()
    if len(frames) < n_frames:
        return None
    return np.stack(frames[:n_frames], axis=0)   # (10, 224, 224, 3)

def list_video_dataset(root, classes):
    paths, labels = [], []
    for cls in classes:
        cls_dir = os.path.join(root, cls)
        if not os.path.isdir(cls_dir):
            print(f"Warning: missing folder for class '{cls}' — skipping")
            continue
        for fname in os.listdir(cls_dir):
            if fname.lower().endswith((".avi", ".mp4", ".mov")):
                paths.append(os.path.join(cls_dir, fname))
                labels.append(cls)
    return paths, labels

video_paths, video_labels = list_video_dataset(VIDEO_ROOT, SELECTED_CLASSES)
print(f"Found {len(video_paths)} videos across {len(set(video_labels))} classes")

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

cnn_extractor = MobileNetV2(weights="imagenet", include_top=False, pooling="avg")
cnn_extractor.trainable = False   # freeze — feature extractor only
FEATURE_DIM = cnn_extractor.output_shape[-1]
print("CNN feature dimension D =", FEATURE_DIM)

def extract_video_features(paths):
    all_feats = []
    for p in paths:
        frames = sample_frames(p)
        if frames is None:
            all_feats.append(None)
            continue
        batch = preprocess_input(frames.astype(np.float32))
        feats = cnn_extractor.predict(batch, verbose=0)   # (10, D)
        all_feats.append(feats)
    return all_feats

if len(video_paths) > 0:
    raw_features = extract_video_features(video_paths)
    keep = [i for i, f in enumerate(raw_features) if f is not None]
    X_video = np.stack([raw_features[i] for i in keep], axis=0)   # (B, 10, D)
    y_video_labels = [video_labels[i] for i in keep]

    vid_le = LabelEncoder()
    y_video = vid_le.fit_transform(y_video_labels)

    print("Video feature tensor shape supplied to RNN:", X_video.shape)
else:
    print("No videos found — populate VIDEO_ROOT with a UCF101 subset before running this cell.")

In [ ]:
if len(video_paths) > 0:
    Xv_train, Xv_test, yv_train, yv_test = train_test_split(
        X_video, y_video, test_size=0.2, stratify=y_video, random_state=42
    )
    Xv_train, Xv_val, yv_train, yv_val = train_test_split(
        Xv_train, yv_train, test_size=0.2, stratify=yv_train, random_state=42
    )

    def build_video_model(cell_type="LSTM", units=32, num_classes=len(vid_le.classes_)):
        RecurrentLayer = layers.LSTM if cell_type == "LSTM" else layers.GRU
        model = models.Sequential([
            layers.Input(shape=(FRAMES_PER_VIDEO, FEATURE_DIM)),
            RecurrentLayer(units),
            layers.Dropout(0.3),
            layers.Dense(num_classes, activation="softmax"),
        ])
        model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        return model

    video_model = build_video_model("LSTM")
    video_history = video_model.fit(
        Xv_train, yv_train, validation_data=(Xv_val, yv_val),
        epochs=20, batch_size=8, verbose=1,
    )

    yv_pred = np.argmax(video_model.predict(Xv_test, verbose=0), axis=1)
    print("Video test accuracy:", accuracy_score(yv_test, yv_pred))
    print(classification_report(yv_test, yv_pred, target_names=vid_le.classes_))
else:
    print("Skipping training — no video data loaded yet.")

In [ ]:
# Plot 7 — sample frames from one video
if len(video_paths) > 0:
    sample = sample_frames(video_paths[0])
    fig, axes = plt.subplots(1, FRAMES_PER_VIDEO, figsize=(20, 3))
    for i, ax in enumerate(axes):
        ax.imshow(sample[i])
        ax.axis("off")
    plt.suptitle(f"Sampled frames — {video_labels[0]}")
    plt.show()

    # Plot 8 — training curves
    h = video_history.history
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(h["loss"], label="Train"); axes[0].plot(h["val_loss"], label="Val")
    axes[0].set_title("Video model — Loss"); axes[0].legend()
    axes[1].plot(h["accuracy"], label="Train"); axes[1].plot(h["val_accuracy"], label="Val")
    axes[1].set_title("Video model — Accuracy"); axes[1].legend()
    plt.show()

    # Plot 9 — confusion matrix
    cm = confusion_matrix(yv_test, yv_pred)
    ConfusionMatrixDisplay(cm, display_labels=vid_le.classes_).plot(xticks_rotation=45)
    plt.title("Video Confusion Matrix")
    plt.show()

    # Example prediction print-out (Section 19 format)
    probs = video_model.predict(Xv_test[:1], verbose=0)[0]
    pred_idx = np.argmax(probs)
    print("Predicted class :", vid_le.inverse_transform([pred_idx])[0])
    print("Actual class    :", vid_le.inverse_transform([yv_test[0]])[0])
    print(f"Confidence      : {probs[pred_idx]:.2f}")

In [ ]:
VOCAB_MIN, VOCAB_MAX = 1, 9        # integers 1..9 as tokens
SEQ_LEN_S2S = 4
N_SAMPLES = 5000

def generate_reversal_data(n_samples, seq_len, vocab_min, vocab_max):
    X = np.random.randint(vocab_min, vocab_max + 1, size=(n_samples, seq_len))
    y = X[:, ::-1]
    return X, y

X_seq, y_seq = generate_reversal_data(N_SAMPLES, SEQ_LEN_S2S, VOCAB_MIN, VOCAB_MAX)
print("Example:", X_seq[0], "->", y_seq[0])

# one-hot encode tokens (vocab 0..9, 0 reserved/pad)
VOCAB_SIZE = VOCAB_MAX + 1

def one_hot(seqs, vocab_size):
    return tf.keras.utils.to_categorical(seqs, num_classes=vocab_size)

X_seq_oh = one_hot(X_seq, VOCAB_SIZE)
y_seq_oh = one_hot(y_seq, VOCAB_SIZE)

Xs_train, Xs_test, ys_train, ys_test = train_test_split(X_seq_oh, y_seq_oh, test_size=0.2, random_state=42)
Xs_train, Xs_val, ys_train, ys_val = train_test_split(Xs_train, ys_train, test_size=0.2, random_state=42)
print(Xs_train.shape, ys_train.shape)

In [ ]:
# Simple encoder-decoder (RepeatVector variant — no teacher forcing, easiest to train for a lab demo)
latent_dim = 64

seq2seq_model = models.Sequential([
    layers.Input(shape=(SEQ_LEN_S2S, VOCAB_SIZE)),
    layers.LSTM(latent_dim, name="encoder_lstm"),          # Encoder -> context vector
    layers.RepeatVector(SEQ_LEN_S2S),                       # Context repeated for each output step
    layers.LSTM(latent_dim, return_sequences=True, name="decoder_lstm"),  # Decoder
    layers.TimeDistributed(layers.Dense(VOCAB_SIZE, activation="softmax")),
])

seq2seq_model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
seq2seq_model.summary()

seq2seq_history = seq2seq_model.fit(
    Xs_train, ys_train, validation_data=(Xs_val, ys_val),
    epochs=40, batch_size=64, verbose=1,
)

In [ ]:
def token_and_sequence_accuracy(model, X, y_true_int):
    y_pred_prob = model.predict(X, verbose=0)
    y_pred = np.argmax(y_pred_prob, axis=-1)
    token_acc = (y_pred == y_true_int).mean()
    seq_acc = (y_pred == y_true_int).all(axis=1).mean()
    return token_acc, seq_acc, y_pred

# Recover integer targets for the test split
ys_test_int = np.argmax(ys_test, axis=-1)
token_acc, seq_acc, y_pred_test = token_and_sequence_accuracy(seq2seq_model, Xs_test, ys_test_int)

print(f"Token Accuracy    : {token_acc*100:.2f}%")
print(f"Sequence Accuracy : {seq_acc*100:.2f}%")
print(f"Final Train Loss  : {seq2seq_history.history['loss'][-1]:.4f}")
print(f"Final Val Loss    : {seq2seq_history.history['val_loss'][-1]:.4f}")

print("\nSample predictions:")
Xs_test_int = np.argmax(Xs_test, axis=-1)
for i in range(5):
    print(f"Input: {Xs_test_int[i]}  ->  Predicted: {y_pred_test[i]}  (Actual: {ys_test_int[i]})")

In [ ]:
consolidated = results_df.copy()
consolidated.loc["Seq2Seq (token/seq acc)"] = [
    token_acc * 100, np.nan, np.nan, seq_acc * 100,
    seq2seq_model.count_params(),
    np.nan,
]
consolidated